# ਪਾਠ 10 - ਉਤਪਾਦਨ ਵਿੱਚ ਏਆਈ ਏਜੰਟ

ਇਸ ਪਾਠ ਵਿੱਚ ਤੁਸੀਂ Microsoft Agent Framework (Python) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਏਆਈ ਏਜੰਟਾਂ ਲਈ **ਉਤਪਾਦਨ ਪੈਟਰਨ** ਸਿੱਖੋਗੇ। ਅਸੀਂ ਕਵਰ ਕਰਦੇ ਹਾਂ:

- **ਨਿਰੀਖਣਯੋਗਤਾ** — ਏਜੰਟ ਇੰਟਰੈਕਸ਼ਨਾਂ ਵਿੱਚ ਟਾਈਮਿੰਗ ਅਤੇ ਲੌਗਿੰਗ ਜੋੜਨਾ
- **ਮੁਲਾਂਕਣ** — ਪ੍ਰਤੀਕਿਰਿਆ ਦੀ ਗੁਣਵੱਤਾ ਨੂੰ ਸਕੋਰ ਕਰਨ ਲਈ ਇੱਕ ਮੁਲਾਂਕਣ ਏਜੰਟ ਦੀ ਵਰਤੋਂ ਕਰਨਾ
- **ਲਾਗਤ ਪ੍ਰਬੰਧਨ** — ਟੋਕਨ ਅਪਟੀਮਾਈਜ਼ੇਸ਼ਨ ਅਤੇ ਮਾਡਲ ਚੋਣ ਲਈ ਰਣਨੀਤੀਆਂ

ਦ੍ਰਿਸ਼ ਇੱਕ **ਯਾਤਰਾ ਏਜੰਟ** ਹੈ ਜੋ ਉਪਭੋਗਤਾਵਾਂ ਨੂੰ ਯਾਤਰਾਵਾਂ ਦੀ ਯੋਜਨਾ ਬਣਾਉਣ ਵਿੱਚ ਮਦਦ ਕਰਦਾ ਹੈ, ਜਿਸ 'ਤੇ ਨਿਗਰਾਨੀ ਅਤੇ ਮੁਲਾਂਕਣ ਲਗਾਏ ਗਏ ਹਨ।


## ਸੈਟਅਪ


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## ਉਤਪਾਦਨ ਵਿਚਾਰ

AI ਏਜੰਟਾਂ ਨੂੰ ਪ੍ਰੋਟੋਟਾਈਪ ਤੋਂ ਉਤਪਾਦਨ ਵਿੱਚ ਲਿਜਾਣ ਲਈ ਤਿੰਨ ਬੁਨਿਆਦੀ ਸਤੰਭਾਂ ਉੱਤੇ ਧਿਆਨ ਦੀ ਲੋੜ ਹੁੰਦੀ ਹੈ:

1. **ਦੇਖਣਯੋਗਤਾ** — ਤੁਹਾਨੂੰ ਇਹ ਦੇਖਣ ਦੀ ਲੋੜ ਹੈ ਕਿ ਏਜੰਟ ਕੀ ਕਰ ਰਿਹਾ ਹੈ, ਇਸ ਵਿੱਚ ਕਿੰਨਾ ਸਮਾਂ ਲੱਗਦਾ ਹੈ, ਅਤੇ ਇਹ ਕਿਹੜੇ ਟੂਲਾਂ ਨੂੰ ਕਾਲ ਕਰਦਾ ਹੈ। ਟ੍ਰੇਸਿੰਗ ਅਤੇ ਲੌਗਿੰਗ ਦੇ ਬਿਨਾਂ, ਪ੍ਰੋਡਕਸ਼ਨ ਦੀਆਂ ਸਮੱਸਿਆਵਾਂ ਨੂੰ ਡੀਬੱਗ ਕਰਨਾ ਲਗਭਗ ਅਸੰਭਵ ਹੈ।

2. **ਮੂਲਾਂਕਣ** — ਸਵੈਚਾਲਿਤ ਗੁਣਵੱਤਾ ਜਾਂਚਾਂ ਇਹ ਯਕੀਨੀ ਬਣਾਉਂਦੀਆਂ ਹਨ ਕਿ ਏਜੰਟ ਦੇ ਜਵਾਬ ਸਮੇਂ ਦੇ ਨਾਲ ਸਹੀ, ਪੂਰੇ ਅਤੇ ਮਦਦਗਾਰ ਰਹਿੰਦੇ ਹਨ। ਇੱਕ ਮੂਲਾਂਕਣ ਏਜੰਟ ਨਿਰਧਾਰਿਤ ਮਾਪਦੰਡਾਂ ਦੇ ਮੁਕਾਬਲੇ ਜਵਾਬਾਂ ਨੂੰ ਸਕੋਰ ਕਰ ਸਕਦਾ ਹੈ।

3. **ਲਾਗਤ ਪ੍ਰਬੰਧਨ** — ਟੋਕਨ ਦੀ ਵਰਤੋਂ ਸਿੱਧਾ ਖ਼ਰਚ 'ਤੇ ਅਸਰ ਪਾਉਂਦੀ ਹੈ। ਪ੍ਰੰਪਟ ਅਨੁਕੂਲਤਾ, ਮਾਡਲ ਚੋਣ, ਅਤੇ ਕੈਸ਼ਿੰਗ ਵਰਗੀਆਂ ਰਣਨੀਤੀਆਂ ਗੁਣਵੱਤਾ ਨੂੰ ਨੁਕਸਾਨ ਪਹੁੰਚਾਏ ਬਿਨਾਂ ਖ਼ਰਚੇ ਨੂੰ ਨਿਯੰਤਰਿਤ ਰੱਖਣ ਵਿੱਚ ਮਦਦ ਕਰਦੀਆਂ ਹਨ।


## ਇੱਕ ਨਿਰੀਖਣਯੋਗ ਏਜੰਟ ਬਣਾਉਣਾ

ਅਸੀਂ ਯਾਤਰਾ ਦੇ ਟੂਲ ਪਰਿਭਾਸ਼ਿਤ ਕਰਦੇ ਹਾਂ ਅਤੇ ਲੈਟੈਂਸੀ ਦੀ ਨਿਗਰਾਨੀ ਕਰਨ ਲਈ ਏਜੰਟ ਕਾਲ ਨੂੰ ਟਾਈਮਿੰਗ ਨਾਲ ਘੇਰਦੇ ਹਾਂ। ਉਤਪਾਦਨ ਵਿੱਚ ਤੁਸੀਂ OpenTelemetry ਜਾਂ ਇਸੇ ਤਰ੍ਹਾਂ ਦੇ ਕਿਸੇ ਟ੍ਰੇਸਿੰਗ ਬੈਕਇੰਡ ਨਾਲ ਇੰਟੈਗਰੇਟ ਕਰੋਗੇ।


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## ਮੁਲਾਂਕਣ ਦੇ ਨਮੂਨੇ

ਇੱਕ ਆਮ ਉਤਪਾਦਨ ਪੈਟਰਨ ਦੂਜੇ ਏਜੰਟ ਨੂੰ ਇੱਕ **ਮੁਲਾਂਕਣਕਰਤਾ** ਵਜੋਂ ਵਰਤਣ ਦਾ ਹੁੰਦਾ ਹੈ। ਮੁਲਾਂਕਣਕਰਤਾ ਮੁੱਖ ਏਜੰਟ ਦੇ ਜਵਾਬ ਨੂੰ ਪਹਿਲਾਂ ਤੋਂ ਨਿਰਧਾਰਿਤ ਮਾਪਦੰਡਾਂ—ਜਿਵੇਂ ਕਿ ਪੂਰਨਤਾ, ਸਹੀਤਾ ਅਤੇ ਮਦਦਗਾਰਤਾ—ਦੇ ਖਿਲਾਫ ਅੰਕ ਦਿੰਦਾ ਹੈ।

ਇਸ ਨਾਲ ਇਹ ਸੰਭਵ ਹੁੰਦਾ ਹੈ:
- ਉਪਭੋਗਤਿਆਂ ਤੱਕ ਜਵਾਬ ਪਹੁੰਚਣ ਤੋਂ ਪਹਿਲਾਂ ਸੁਚਾਲਿਤ ਗੁਣਵੱਤਾ ਨਿਰੀਖਣ
- ਜਦੋਂ ਪ੍ਰਾਂਪਟ ਜਾਂ ਮਾਡਲ ਬਦਲਦੇ ਹਨ ਤਾਂ ਰਿਗਰੇਸ਼ਨ ਦੀ ਪਛਾਣ
- ਸਮੇਂ ਦੇ ਨਾਲ ਏਜੰਟ ਦੇ ਪ੍ਰਦਰਸ਼ਨ ਦੀ ਲਗਾਤਾਰ ਨਿਗਰਾਨੀ


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## ਲਾਗਤ ਪ੍ਰਬੰਧਨ ਰਣਨੀਤੀਆਂ

ਉਤਪਾਦਨ AI ਏਜੰਟਾਂ ਲਈ ਖਰਚਾਂ ਨੂੰ ਕੰਟਰੋਲ ਕਰਨਾ ਬਹੁਤ ਜ਼ਰੂਰੀ ਹੈ। ਇੱਥੇ ਕੁਝ ਮੁੱਖ ਰਣਨੀਤੀਆਂ ਹਨ:

| Strategy | Description |
|---|---|
| **Prompt optimization** | ਸਿਸਟਮ ਦੀਆਂ ਹਦਾਇਤਾਂ ਸੰਖੇਪ ਰੱਖੋ। ਇਨਪੁਟ ਟੋਕਨ ਘਟਾਉਣ ਲਈ ਅਣਜਰੂਰੀ ਸੰਦਰਭ ਹਟਾਓ। |
| **Model selection** | ਸਰਲ ਕਾਰਜਾਂ ਲਈ ਛੋਟੇ, ਸਸਤੇ ਮਾਡਲ (ਉਦਾਹਰਨ ਲਈ GPT-4o-mini) ਵਰਤੋ ਜਿਵੇਂ ਕਿ ਵਰਗੀਕਰਨ ਜਾਂ ਨਿਕਾਸ, ਅਤੇ ਜਟਿਲ ਤਰਕ ਲਈ ਵੱਡੇ ਮਾਡਲ ਰੱਖੋ। |
| **Caching** | ਮੁੜ-ਹੋਣ ਵਾਲੀਆਂ API ਕਾਲਾਂ ਤੋਂ ਬਚਣ ਲਈ ਟੂਲ ਦੇ ਨਤੀਜੇ ਅਤੇ ਅਕਸਰ ਹੋਣ ਵਾਲੀਆਂ ਪੁੱਛਗਿੱਛਾਂ ਨੂੰ ਕੇਸ਼ ਕਰੋ। |
| **Token budgets** | `max_tokens` ਦੀਆਂ ਹੱਦਾਂ ਨਿਰਧਾਰਤ ਕਰੋ ਤਾਂ ਜੋ ਅਣਉਮੀਦ ਤੌਰ 'ਤੇ ਲੰਮੇ ਜਵਾਬਾਂ ਤੋਂ ਬਚਿਆ ਜਾ ਸਕੇ। |
| **Batching** | ਸੰਭਵ ਹੋਵੇ ਤਾਂ ਕਈ ਯੂਜ਼ਰ ਪੁੱਛਤਾਛਾਂ ਨੂੰ ਇੱਕ ਇਕੱਲੀ API ਕਾਲ ਵਿੱਚ ਜੋੜੋ। |

ਅਮਲ ਵਿੱਚ, ਇੱਕ ਪੱਧਰੀ ਪਹੁੰਚ ਚੰਗੀ ਤਰ੍ਹਾਂ ਕੰਮ ਕਰਦੀ ਹੈ: ਸਰਲ ਬੇਨਤੀਆਂ ਨੂੰ ਇੱਕ ਤੇਜ਼, ਸਸਤੇ ਮਾਡਲ ਵੱਲ ਭੇਜੋ ਅਤੇ ਸਿਰਫ਼ ਜਟਿਲ ਪੁੱਛਤਾਛਾਂ ਨੂੰ ਹੀ ਵਧੇਰੇ ਸਮਰੱਥ ਮਾਡਲ ਤੱਕ ਭੇਜੋ।


## ਸਾਰ

ਇਸ ਪਾਠ ਵਿੱਚ ਤੁਸੀਂ ਇਹ ਸਿੱਖਿਆ ਕਿ:

1. **ਦੇਖਣਯੋਗਤਾ ਸ਼ਾਮਲ ਕਰੋ** ਏਜੰਟ ਇੰਟਰੈਕਸ਼ਨਾਂ ਵਿੱਚ ਸਮੇਂ ਅਤੇ ਲੌਗਿੰਗ ਦੇ ਨਾਲ, ਟ੍ਰੇਸਿੰਗ ਅਤੇ ਮਾਨੀਟਰਿੰਗ ਲਈ ਨੀਵ ਰੱਖਦਿਆਂ।
2. **ਏਜੰਟ ਦੇ ਜਵਾਬਾਂ ਦਾ ਮੂਲਿਆਂਕਣ ਕਰੋ** ਇੱਕ ਮੁਲਾਂਕਣ ਕਰਨ ਵਾਲੇ ਏਜੰਟ ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਆਟੋਮੈਟਿਕ ਤੌਰ 'ਤੇ ਜੋ ਪੂਰਨਤਾ, ਸ਼ੁੱਧਤਾ ਅਤੇ ਸਹਾਇਕਤਾ ਨੂੰ ਸਕੋਰ ਕਰਦਾ ਹੈ।
3. **ਖਰਚਿਆਂ ਦਾ ਪ੍ਰਬੰਧ ਕਰੋ** ਪ੍ਰੌਂਪਟ ਅਪਟੀਮਾਈਜ਼ੇਸ਼ਨ, ਮਾਡਲ ਚੋਣ, ਕੈਸ਼ਿੰਗ ਅਤੇ ਟੋਕਨ ਬਜਟਾਂ ਰਾਹੀਂ।

ਇਹ ਪ੍ਰੋਡਕਸ਼ਨ ਪੈਟਰਨ ਤੁਹਾਡੇ AI ਏਜੰਟਾਂ ਨੂੰ ਵੱਡੇ ਪੱਧਰ 'ਤੇ ਭਰੋਸੇਯੋਗ, ਮਾਪਯੋਗ ਅਤੇ ਲਾਗਤ-ਕੁਸ਼ਲ ਬਣਾਉਣ ਵਿੱਚ ਮਦਦ ਕਰਦੇ ਹਨ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
ਅਸਵੀਕਾਰ:
ਇਹ ਦਸਤਾਵੇਜ਼ AI ਅਨੁਵਾਦ ਸੇਵਾ Co-op Translator (https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਅਨੁਵਾਦ ਕੀਤਾ ਗਿਆ ਹੈ। ਅਸੀਂ ਸਹੀ ਹੋਣ ਲਈ ਕੋਸ਼ਿਸ਼ ਕਰਦੇ ਹਾਂ, ਪਰ ਕਿਰਪਾ ਕਰਕੇ ਧਿਆਨ ਰੱਖੋ ਕਿ ਆਟੋਮੇਟਿਕ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਤ੍ਰੁੱਟੀਆਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਮੂਲ ਦਸਤਾਵੇਜ਼ ਆਪਣੀ ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਹੀ ਅਧਿਕਾਰਕ ਸਰੋਤ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਮਹੱਤਵਪੂਰਣ ਜਾਣਕਾਰੀ ਲਈ, ਪੇਸ਼ੇਵਰ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫ਼ਾਰਿਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਅਸੀਂ ਇਸ ਅਨੁਵਾਦ ਦੇ ਉਪਯੋਗ ਕਾਰਨ ਹੋਣ ਵਾਲੀਆਂ ਕਿਸੇ ਵੀ ਗਲਤਫਹਿਮੀਆਂ ਜਾਂ ਗਲਤ ਵਿਆਖਿਆਵਾਂ ਲਈ ਜ਼ਿੰਮੇਵਾਰ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
